# 推論最適化ラボ：**The Margin Call**

あなたのファームは**Project HELVETICA**を受注しました。**Aldgate Capital Partners**向けの、AIを活用した契約デューデリジェンス案件です。同社が買収しようとしているVolta Industrial Groupは、14社を統合したグループで、サプライヤー契約は**248,000件**に上ります。ディールチームは、すべての契約について、支配権変更に伴う同意、賠償責任の上限、自動更新、準拠法を仕分ける必要があります。

デモチームは、ピッチの前夜にエージェント**ClauseScan v0**を作りました。動きます。ただし、考えている間にアナリストが別のタブへ逃げてしまうほど遅く、ユニットエコノミクスは知らないうちに案件マージンを食いつぶしています。ピッチは成功し、SOWも締結され、今朝からこのエージェントは**あなたの担当**です。

お客様と合意したSLAは次のとおりです。

| SLAの項目 | 合意内容 |
|---|---|
| 精度 | 監査対象フィールドで ≥ 90% |
| インタラクティブのレイテンシ | アナリストの作業セッションで、契約あたりp50 ≤ 5s |
| ユニットエコノミクス | 本番規模で、契約あたり売上原価（COGS）≤ $0.02 |
| 料金 | レビュー済み契約1件あたり$0.75（固定） |

**ラボの3幕構成：**
1. **パート1〜2：計測と基準値の確立。** 計測ツールを作り、Haiku / Sonnet / Opusで最初のテストを実行します。
2. **パート3〜4：診断と調整項目の習得。** ClauseScan v0を実行してダメージを確認し、そのあと6つの最適化の調整項目を、1つずつ測定しながら見ていきます。
3. **パート5〜6：最適化スプリント。** パイプラインを作り直し、リーダーボードを駆け上がり、ステアリングコミッティに持っていく変更前後のスライドを自動生成します。

**主要な指標：**
- **TTFT**：最初のトークンまでの時間（Time To First Token）。アナリストがスピナーを見つめて待つ時間です。
- **TTC**：完了までの時間（Time To Completion）。リクエスト全体の所要時間です。
- **OTPS**：秒あたり出力トークン数（Output Tokens Per Second）。ストリーミング開始後の生成スループットです。
- **$/contract**：キャッシュの読み出しと書き込みを含む、ドキュメント1件あたりの実質コストです。

> 💸 このノートブックの全セルを実行すると、API利用料は約**$2〜4**かかります。大半は、わざと無駄の多いv0の基準値測定によるものです。それも学びの一部です。

---

**Partner Basecamp参加者向けに作成。** 研修資料としての複製・再配布は禁止です。ここで学ぶパターンをご自身のお客様案件で活用することは自由です。

# パート0 · セットアップ

まず依存パッケージ、次にAPIキーです。キーをセルに貼り付けてはいけません。いつかこのノートブックをお客様のリポジトリに置く日が来ます。今のうちに習慣にしておきましょう。

In [ ]:
# 依存パッケージを、このカーネルにインストールします。再実行しても安全で、ロックされた（PEP 668）Pythonでも動作します。
import importlib.util, os, subprocess, sys

# ── 環境ガード（Environment guard）─────────────────────────────────
# 下のinstallerと同じセルに置いているので、飛ばすことはできません。
# 素のシステムPythonにパッケージがインストールされることは決してありません（以前の最悪のケースは、
# IT管理のマシンに対する --break-system-packages でした）。ここで止まった場合は、
# SETUP.md の「Why the notebook just stopped」を参照してください。
def _in_isolated_env():
    """実行中のカーネルが venv / virtualenv / conda 環境、またはColabであればTrueを返します。
    判定はインタプリタ自身（sys.*）から行います。起動元のシェルから引き継いだ
    有効化用の環境変数は、sys.executable が実際にその環境の内側にある場合にのみ
    信頼します。有効化済みのターミナルから起動したシステムPythonのカーネルも
    VIRTUAL_ENV を引き継ぐため、それだけで通してはいけません。"""
    if "google.colab" in sys.modules:
        return True  # Colabは使い捨てのランタイムを自前でサンドボックス化している
    if sys.prefix != getattr(sys, "base_prefix", sys.prefix):
        return True  # PEP 405 venv（python -m venv）。ほとんどのconda環境も該当
    if hasattr(sys, "real_prefix"):
        return True  # 旧来のvirtualenv
    exe = os.path.realpath(sys.executable)
    for var in ("VIRTUAL_ENV", "CONDA_PREFIX"):
        root = os.environ.get(var)
        if not root or not exe.startswith(os.path.realpath(root) + os.sep):
            continue  # シェルからの伝聞にすぎない。kernelは別の場所にある
        if var == "CONDA_PREFIX" and os.environ.get("CONDA_DEFAULT_ENV", "base") == "base":
            continue  # condaの共有 `base` は隔離環境とは見なさない
        return True
    return False

if os.environ.get("BASECAMP_ALLOW_SYSTEM_PYTHON") == "1":
    print("⚠️  環境ガードをバイパスしました（BASECAMP_ALLOW_SYSTEM_PYTHON=1）。"
          "意図的にこのPythonへインストールします。")
elif not _in_isolated_env():
    _HEAD = "✗ システムPythonを検出しました。何もインストールせずに停止しました"
    _BODY = (
        "ここにパッケージをインストールすると、マシン全体のPythonが変わります。\n"
        "会社管理のノートPCでは、ITチケットが必要になることがあります。\n"
        "\n"
        "対処方法（2ステップ）：\n"
        "  1. ターミナルで、リポジトリのルートから実行します：\n"
        "       python3 -m venv .venv\n"
        "       source .venv/bin/activate          # Windows: .venv\\Scripts\\activate\n"
        "       pip install -r requirements.txt\n"
        "  2. VS Codeで、カーネル名（右上）をクリック → Select Another Kernel →\n"
        "     Python Environments → 末尾が .venv のものを選択 → このセルをもう一度実行します。\n"
        "\n"
        "condaを使っている場合は、`conda activate <env>`（`base` 以外）を実行してから、そのカーネルを選びます。\n"
        "自己管理のマシンをお使いのファシリテーターの方へ：BASECAMP_ALLOW_SYSTEM_PYTHON=1 でバイパスできます。"
    )
    _shown = False
    try:  # 下のAPIキーチェックと同じバナー表現（緑のボックスの赤いバリアント）
        from IPython import get_ipython
        if get_ipython().__class__.__name__ == "ZMQInteractiveShell":
            import html as _html
            from IPython.display import HTML, display
            display(HTML(
                '<div style="padding:12px 16px;border-radius:8px;background:#fdecea;'
                'border:1.5px solid #b42318;font-size:15px;font-family:sans-serif;">'
                '<div style="color:#b42318;font-weight:600;">' + _html.escape(_HEAD) + '</div>'
                '<pre style="margin:10px 0 0;font-family:inherit;font-size:14px;font-weight:400;'
                'color:#141413;white-space:pre-wrap;">' + _html.escape(_BODY) + '</pre></div>'
            ))
            _shown = True
    except Exception:
        pass
    raise SystemExit(
        "環境ガードがこのセルを停止しました。上のメッセージを確認してください。"
        if _shown else "\n  " + _HEAD + "\n\n" + _BODY + "\n"
    )
else:
    print("✓ 環境ガード：隔離されたインタプリタを検出しました。インストールしても安全です")
# ──────────────────────────────────────────────────────────────────────────


def _ensure_packages(requirements):
    """requirements: (import_name, pip_spec) のリスト。足りないものだけを、実行中の
    インタプリタにインストールします。まず通常のインストール、次にユーザー領域、
    最後にPEP 668の上書き（ユーザー領域が先、システム全体は最後の手段）を試します。
    どの試行もサイレントで、pipの出力は表示せず取り込むだけなので、ロックされた
    Python（HomebrewやDebian、PEP 668）でも、フォールバックが実際に成功したときに
    'externally-managed-environment' という恐ろしい文面が大量に出ることはありません。
    すべての方法が失敗したときだけ、生のトレースバックではなく、venvによる
    解決方法とともに理由を表示します。"""
    missing = [pip for mod, pip in requirements if importlib.util.find_spec(mod) is None]
    if not missing:
        return
    print("Installing " + ", ".join(missing) + " （初回のみ、しばらくお待ちください）…", flush=True)
    base = [sys.executable, "-m", "pip", "install", "-q"]
    last = None
    for extra in ([], ["--user"], ["--user", "--break-system-packages"], ["--break-system-packages"]):
        last = subprocess.run(base + extra + missing, capture_output=True, text=True)
        if last.returncode == 0:
            return
    pip_said = (last.stderr or last.stdout or "").strip().splitlines() if last else []
    tail = "\n      ".join(pip_said[-3:]) if pip_said else "（pipの出力なし）"
    raise SystemExit(
        "\n  インストールできませんでした： " + ", ".join(missing) + "\n"
        "  このPythonはロックされている（PEP 668）か、オフラインです。最も簡単な解決策はvenvです：\n"
        f"      {sys.executable} -m venv .venv\n"
        "      source .venv/bin/activate          # Windowsの場合はSETUP.mdを参照\n"
        f"      pip install {' '.join(missing)}\n"
        "  その後、VS Code（右上のカーネルピッカー）で.venvのインタプリタを選び、Run Allを実行してください。\n"
        "  社内プロキシでPyPIがブロックされている場合は、リポジトリ直下のSETUP.mdを参照してください。\n"
        f"  (pip said: {tail})\n"
    )

_ensure_packages([("anthropic", "anthropic"), ("tabulate", "tabulate")])
print("✓ 依存パッケージの準備ができました")

import os
import json
import time
import statistics
from dataclasses import dataclass, field
from typing import List, Optional
from concurrent.futures import ThreadPoolExecutor
from tabulate import tabulate

import anthropic

### セットアップ：Claudeに接続する

次のセルを最初に実行してください。セットアップセルは、初回実行時に**`.env`ファイル**を作成します（gitignore済みなので、キーがコミットされることはありません）。ファイルを開き、`ANTHROPIC_API_KEY=`の後ろにキーを貼り付けて保存し、再実行してください。カーネルを再起動しても残るので、貼り付けるのは1回だけです。*（まだ`.env`がない場合は、代わりに非表示の入力ボックスが表示されます。）* 緑色の**「✓ APIキーを確認しました」**バナーが表示されれば完了です。赤いバナーが出たら、書かれている指示に従って、もう一度セルを実行してください。

In [ ]:
import os

def _status(ok, msg):
    """ノートブックでは緑 / 赤のバナー、スクリプトとして実行したときはプレーンテキスト。"""
    try:
        from IPython import get_ipython
        shell = get_ipython()
        if shell is None or shell.__class__.__name__ != "ZMQInteractiveShell":
            raise RuntimeError("ノートブックカーネルではありません。プレーンテキストのバナーを使います")
        from IPython.display import display, HTML
        color = "#1a7f37" if ok else "#b42318"
        bg = "#e6f4ea" if ok else "#fdecea"
        icon = "✓" if ok else "✗"
        display(HTML(
            f'<div style="padding:12px 16px;border-radius:8px;background:{bg};'
            f'border:1.5px solid {color};color:{color};font-weight:600;'
            f'font-size:15px;font-family:sans-serif;">{icon} {msg}</div>'
        ))
    except Exception:
        print(("[OK] " if ok else "[!!] ") + msg)

import os
import pathlib

import anthropic

# ── Claudeに接続する：Anthropic API または Amazon Bedrock ──
# どちらの認証情報（credential）でも動作します。セルが、どちらをお持ちか自動で判別します。
#   Anthropic API : ANTHROPIC_API_KEY=sk-ant-...
#   Amazon Bedrock: AWS_BEARER_TOKEN_BEDROCK=...  と  AWS_REGION=us-east-1
# お使いの方を、このセルが作成する.envファイルに書いてください（gitignore済みで、コミットされません）。
# またはシェルでexportしても構いません。シェルの値は.envファイルより優先されます。
_ENV_TEMPLATE = (
    "# Anthropic API key — paste after the = (no quotes, no spaces), then save and\n"
    "# re-run the setup cell. Get one at https://console.anthropic.com/\n"
    "ANTHROPIC_API_KEY=paste-your-key-here\n"
    "\n"
    "# --- Using Amazon Bedrock instead? Comment out the line above and fill these in:\n"
    "# AWS_BEARER_TOKEN_BEDROCK=paste-your-bedrock-api-key-here\n"
    "# AWS_REGION=us-east-1\n"
)


def _resolve_env_file():
    """最も近い既存の.envを、作業ディレクトリから上へたどって探します（ルートの.env 1つで
    すべての演習をまかなえます）。まだ無ければ、リポジトリのルートを指します。
    ノートブックが単独で開かれた場合は、このフォルダーを指します。"""
    here = pathlib.Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / ".env").is_file():
            return d / ".env"
    root = next((d for d in [here, *here.parents]
                 if (d / "SETUP.md").exists() or (d / ".git").exists()), here)
    return root / ".env"


_env_file = _resolve_env_file()
if not _env_file.exists():
    _env_file.write_text(_ENV_TEMPLATE)
    print(f"Created {_env_file.name} in {_env_file.parent} — open it, add your key, "
          "保存してから、このセルをもう一度実行してください。")

# 小さな.envパーサー（python-dotenv不要）。実行のたびに読み直すので、キーを貼り付けて
# 再実行すれば反映されます。環境（シェル / Claude Code / CI）に実際の値があれば、
# それが優先され、プレースホルダー（placeholder）が残ることはありません。
_file = {}
for _line in (_env_file.read_text().splitlines() if _env_file.exists() else []):
    _line = _line.strip()
    if _line and not _line.startswith("#") and "=" in _line:
        _k, _v = _line.split("=", 1)
        _file[_k.strip()] = _v.strip().strip('"').strip("'")
for _k, _v in _file.items():
    if _k != "ANTHROPIC_API_KEY":
        os.environ.setdefault(_k, _v)

_shell_key = os.environ.get("ANTHROPIC_API_KEY", "").strip()
_anthropic_key = _shell_key if _shell_key.startswith("sk-ant-") else _file.get("ANTHROPIC_API_KEY", "").strip()
_bedrock_token = os.environ.get("AWS_BEARER_TOKEN_BEDROCK", "").strip()
_bedrock_region = os.environ.get("AWS_REGION", "").strip()

if _anthropic_key.startswith("sk-ant-"):
    PROVIDER = "anthropic"
elif _bedrock_token:
    PROVIDER = "bedrock"
else:
    PROVIDER = None


def _needs_credentials(head, body):
    """警告の黄色いバナーを出して停止します。数セル先ではなく、ここで失敗させるためです。"""
    _shown = False
    try:
        from IPython import get_ipython
        if get_ipython().__class__.__name__ == "ZMQInteractiveShell":
            import html as _html
            from IPython.display import HTML, display
            display(HTML(
                '<div style="padding:12px 16px;border-radius:8px;background:#fff8c5;'
                'border:1.5px solid #9a6700;font-size:15px;font-family:sans-serif;">'
                '<div style="color:#9a6700;font-weight:600;">' + _html.escape(head) + '</div>'
                '<pre style="margin:10px 0 0;font-family:inherit;font-size:14px;font-weight:400;'
                'color:#141413;white-space:pre-wrap;">' + _html.escape(body) + '</pre></div>'
            ))
            _shown = True
    except Exception:
        pass
    if not _shown:
        print("\n" + head + ":\n   " + body.replace("\n", "\n   ") + "\n")
    raise SystemExit("認証情報がありません。上のメッセージを確認してください。")


if PROVIDER is None:
    _needs_credentials(
        "📋 続行するには認証情報を追加してください",
        f"Open this file:  {_env_file}\n"
        "\n"
        "Anthropic APIを使う場合は、次を設定してください：\n"
        "    ANTHROPIC_API_KEY=sk-ant-...\n"
        "\n"
        "Amazon Bedrockを使う場合は、両方を設定してください：\n"
        "    AWS_BEARER_TOKEN_BEDROCK=<your Bedrock API key>\n"
        "    AWS_REGION=us-east-1          # モデルを有効化しているリージョン\n"
        "\n"
        "ファイルを保存してから、このセルの ▶ をもう一度クリックしてください。"
    )

if PROVIDER == "bedrock" and not _bedrock_region:
    _needs_credentials(
        "📋 Bedrockにはリージョンの指定が必要です",
        f"Found AWS_BEARER_TOKEN_BEDROCK but no AWS_REGION.\n"
        f"\n"
        f"Open this file:  {_env_file}\n"
        "Bedrockのモデルを有効化しているリージョンを追加してください。例：\n"
        "    AWS_REGION=us-east-1\n"
        "\n"
        "ファイルを保存してから、このセルの ▶ をもう一度クリックしてください。"
    )


def _model(name):
    """BedrockのモデルIDには `anthropic.` という接頭辞が付きます。Anthropic APIでは接頭辞なしのIDを使います。"""
    return f"anthropic.{name}" if PROVIDER == "bedrock" else name


# この演習で使うモデルに名前を付けます。使用しているプロバイダー（provider）に合わせて解決されます。
MODEL = _model("claude-sonnet-5")        # この演習のworkhorse（主力）
FAST_MODEL = _model("claude-haiku-4-5")  # 安くて速い（接続確認、judge用）
BIG_MODEL = _model("claude-opus-4-8")    # より大きなモデル（larger model）を試したいとき


def _make_client(timeout, max_retries=2):
    if PROVIDER == "bedrock":
        from anthropic import AnthropicBedrockMantle
        return AnthropicBedrockMantle(aws_region=_bedrock_region,
                                      timeout=timeout, max_retries=max_retries)
    return anthropic.Anthropic(api_key=_anthropic_key,
                               timeout=timeout, max_retries=max_retries)


# Connection check：認証情報と、このモデルが使えることの両方を検証します。
# Bedrockでは、キーが有効でも、アカウントやリージョンでモデルが有効化されていなければ404になることがあります。
# そのため、認証情報の形式を確認するだけでなく、実際のモデルIDにpingを送ります。
_probe = _make_client(timeout=30.0, max_retries=1)
try:
    _probe.messages.create(model=FAST_MODEL, max_tokens=1,
                           messages=[{"role": "user", "content": "ping"}])
except anthropic.NotFoundError:
    if PROVIDER == "bedrock":
        _status(False, f"Bedrock reached, but model '{FAST_MODEL}' isn't available to you in "
                       f"{_bedrock_region}. Enable model access for it in the Bedrock console "
                       f"(or switch AWS_REGION to a region where it is enabled), then re-run.")
    else:
        _status(False, f"Model '{FAST_MODEL}' not found for this key.")
    raise SystemExit("モデルが利用できません。上のメッセージを確認してください。")
except (anthropic.AuthenticationError, anthropic.PermissionDeniedError):
    if PROVIDER == "bedrock":
        _status(False, "そのBedrockキーは拒否されました。AWS_BEARER_TOKEN_BEDROCKを確認し、"
                       "Bedrockの呼び出し権限があることも確認してから、このセルをもう一度実行してください。")
    else:
        _status(False, "そのキーは拒否されました。このセルをもう一度実行し、キー全体を"
                       "貼り付けてください（sk-ant-で始まります）。")
    raise SystemExit("認証情報が受け付けられませんでした。このセルを再実行して、もう一度お試しください。")
except Exception as exc:
    _status(False, "APIに接続できませんでした（" + type(exc).__name__ + "）。ネットワークの"
                   "接続を確認してから、このセルをもう一度実行してください。")
    raise
else:
    if PROVIDER == "anthropic":
        os.environ["ANTHROPIC_API_KEY"] = _anthropic_key  # 以降のセルと !python がこれを拾います
        _status(True, "APIキーを確認しました。Claudeに接続されています。")
    else:
        _status(True, f"Bedrock key verified ({_bedrock_region}) - you're connected to Claude "
                      f"as {MODEL}.")

# 作業用クライアント。非ストリーミング呼び出しでmax_tokens>21333を使うには、長いタイムアウトが必要です。
client = _make_client(timeout=900.0)


In [ ]:
# ラボ用クライアント：SDKのデフォルト（長いタイムアウト、標準のリトライ）。上の検証用クライアントは
# 30秒の短いタイムアウトを使っています。1トークンのpingには十分ですが、v0のわざと遅い、
# 非ストリーミング（non-streaming）の長文呼び出しには致命的です。そこで再作成します。キーはすでに環境にあります。
client = anthropic.Anthropic()

# モデルのポートフォリオ。日付固定のIDではなくエイリアスを使います。エイリアスは現在のスナップショットに
# 追従するので、snapshotが廃止されても404になりません。
MODEL_HAIKU = FAST_MODEL   # セットアップでプロバイダー（Anthropic APIまたはBedrock）に合わせて解決済み
MODEL_SONNET = MODEL
MODEL_OPUS = BIG_MODEL

print(f"SDK {anthropic.__version__} · portfolio: {MODEL_HAIKU}, {MODEL_SONNET}, {MODEL_OPUS}")

# パート1 · 計測ツール

計測できないものは最適化できません。そして、改善を証明できなければ、その改善分をお客様に請求することもできません。このラボのすべての処理は、ストリーミングヘルパー1つと、キャッシュを考慮したコスト関数1つを通ります。

In [ ]:
@dataclass
class BenchmarkResult:
    """1回のAPI呼び出しにおける時間、トークン、コスト。"""
    ttft: float                    # 最初のトークンまでの時間（TTFT、秒）
    total_time: float              # 完了までの時間（TTC、秒）
    input_tokens: int
    output_tokens: int
    model: str
    test_name: str
    otps: Optional[float] = None   # 秒あたり出力トークン数（OTPS）
    cost: Optional[float] = None   # ドル換算、キャッシュ考慮（cache-aware）
    cache_read: int = 0
    cache_write: int = 0


@dataclass
class BenchmarkSuite:
    """実行ごとの結果を集めて、比較表を出力します。"""
    results: List[BenchmarkResult] = field(default_factory=list)

    def add(self, result: BenchmarkResult):
        self.results.append(result)

    def clear(self):
        self.results = []

    def summary(self) -> str:
        if not self.results:
            return "結果がありません。"
        groups: dict = {}
        for r in self.results:
            groups.setdefault(r.test_name, []).append(r)
        rows = []
        for name, group in groups.items():
            rows.append([
                name,
                len(group),
                f"{statistics.mean(r.ttft for r in group) * 1000:.0f}",
                f"{statistics.mean(r.total_time for r in group) * 1000:.0f}",
                f"{statistics.mean(r.otps or 0 for r in group):.1f}",
                f"${sum(r.cost or 0 for r in group) * 1000:.2f}",
            ])
        headers = ["Test", "Runs", "TTFT(ms)", "TTC(ms)", "OTPS", "$/1K calls"]
        return tabulate(rows, headers=headers, tablefmt="grid")


suite = BenchmarkSuite()
print("BenchmarkSuiteの準備ができました")

**ストリーミングヘルパー。** ユーザー向けの処理では、ストリーミングが基本姿勢です。アナリストは応答全体の完了（TTC）を待たずに、トークンをすぐに見られます（TTFT）。長い応答でもHTTPタイムアウトに引っかかりません。ここではイベントストリームを監視し、最初の`content_block_start`でTTFTを記録してから、最終メッセージを集めます。

In [ ]:
def _stream_request(messages, model, max_tokens=1024, system=None, **kwargs):
    """リクエストをストリーミングし、(ttft_seconds, total_seconds, final_message) を返します。"""
    ttft = None
    params = dict(model=model, max_tokens=max_tokens, messages=messages, **kwargs)
    if system is not None:
        params["system"] = system
    start = time.perf_counter()
    with client.messages.stream(**params) as stream:
        for event in stream:
            if ttft is None and event.type == "content_block_start":
                ttft = time.perf_counter() - start
        response = stream.get_final_message()
    total = time.perf_counter() - start
    return ttft if ttft is not None else total, total, response


def compute_otps(ttft, total_time, output_tokens):
    """秒あたり出力トークン数。生成時間（TTCからTTFTを引いた時間）で計測します。
    最初のトークンが出るまでの時間は、生成ではなく待ち時間です。"""
    gen_time = max(total_time - ttft, 1e-9)
    return output_tokens / gen_time, gen_time


def text_of(response) -> str:
    return "".join(b.text for b in response.content if b.type == "text")


ttft, total, resp = _stream_request(
    [{"role": "user", "content": "2 + 2 は？ 一言で答えてください。"}], model=MODEL_SONNET)
otps, gen = compute_otps(ttft, total, resp.usage.output_tokens)
print(f"Response: {text_of(resp).strip()}")
print(f"TTFT {ttft*1000:.0f}ms · TTC {total*1000:.0f}ms · OTPS {otps:.1f} tok/s")

**キャッシュを考慮したコスト。** 世の中の多くのコスト関数は、`input × rate + output × rate`で価格を計算して終わりです。これではキャッシュを使うワークロードのコストを大きく見誤ります。キャッシュの**書き込み**は入力レートの**1.25倍**（TTL 5分）、キャッシュの**読み出し**は**0.1倍**で課金されます。お客様のCFOの前に立つときは、ユニットエコノミクスに4つの項がすべて必要です。

| モデル | 入力 $/MTok | 出力 $/MTok |
|---|---|---|
| Haiku 4.5 | $1.00 | $5.00 |
| Sonnet 5 | $3.00 | $15.00 |
| Opus 4.8 | $5.00 | $25.00 |

*（成果物で価格を引用する前に、料金ページで確認してください。この表は2026年6月時点のキャッシュです。Batch APIはすべてのトークン料金が50%になります。この調整項目はパート4で登場します。）*

In [ ]:
PRICING = {
    MODEL_HAIKU:  {"input": 1.00, "output": 5.00},
    MODEL_SONNET: {"input": 3.00, "output": 15.00},
    MODEL_OPUS:   {"input": 5.00, "output": 25.00},
}
CACHE_WRITE_MULT = 1.25   # 5分TTLのキャッシュ書き込み
CACHE_READ_MULT = 0.10
BATCH_DISCOUNT = 0.50     # Batch API：すべてのトークン料金が50%オフ


def calculate_cost(model: str, usage) -> float:
    """1回のAPI呼び出しの実質コスト（ドル）。キャッシュの経済性を含みます。"""
    p = PRICING[model]
    cache_write = getattr(usage, "cache_creation_input_tokens", 0) or 0
    cache_read = getattr(usage, "cache_read_input_tokens", 0) or 0
    return (
        usage.input_tokens * p["input"]
        + cache_write * p["input"] * CACHE_WRITE_MULT
        + cache_read * p["input"] * CACHE_READ_MULT
        + usage.output_tokens * p["output"]
    ) / 1e6


def measure(prompt: str, model: str, test_name: str, max_tokens: int = 256) -> BenchmarkResult:
    """計測付きのストリーミング呼び出し1回 → BenchmarkResult。"""
    ttft, total, resp = _stream_request(
        [{"role": "user", "content": prompt}], model=model, max_tokens=max_tokens)
    otps, _ = compute_otps(ttft, total, resp.usage.output_tokens)
    return BenchmarkResult(
        ttft=ttft, total_time=total,
        input_tokens=resp.usage.input_tokens, output_tokens=resp.usage.output_tokens,
        model=model, test_name=test_name, otps=otps,
        cost=calculate_cost(model, resp.usage),
        cache_read=getattr(resp.usage, "cache_read_input_tokens", 0) or 0,
        cache_write=getattr(resp.usage, "cache_creation_input_tokens", 0) or 0,
    )


print(f"Cost of the 2+2 call above: ${calculate_cost(MODEL_SONNET, resp.usage):.6f}")

# パート2 · 初期テスト：モデルポートフォリオ

同じプロンプトを3つのモデルで、それぞれ4回ずつ実行します。これは最も効果の大きい調整項目なので、最初に扱います。モデルを*どう*呼び出すかを最適化する前に、仕事の内容ごとに*どの*モデルがふさわしいかを決めましょう。

In [ ]:
PROBE = "機械学習とは何ですか？ 2文で答えてください。"

suite.clear()
for model_id, label in [(MODEL_HAIKU, "haiku"), (MODEL_SONNET, "sonnet"), (MODEL_OPUS, "opus")]:
    print(f"Benchmarking {label}...")
    for i in range(4):
        r = measure(PROBE, model=model_id, test_name=label)
        suite.add(r)
        print(f"  run {i+1}: TTFT {r.ttft*1000:.0f}ms · TTC {r.total_time*1000:.0f}ms"
              f" · OTPS {r.otps:.1f} · ${r.cost:.6f}")

print()
print(suite.summary())

**ベンチマークブログではなく、パートナーの目で表を読みましょう。** Haikuは通常、数倍安く、最初のトークンまでの時間も明らかに短くなります。Opusが提供する深さが必要になるのは、難しいケースだけです。適切なアーキテクチャは、ほとんどの場合「1つを選ぶ」ことではなく、**ポートフォリオ**です。大量の定型業務には安価なモデル、判断が必要な場面には高価なモデル、そしてどちらに振り分けるかを決めるルーター。この考え方を覚えておいてください。調整項目2になります。

# パート3 · 案件：ClauseScan v0を知る

以下は、HELVETICAワークロードのラボ規模版です。Volta傘下のサプライヤー契約から抜き出した6件のサンプル（本番の契約は約8倍の長さです。その補正はパート6で正直に行います）、全レビュアーが従う基準書である**デューデリジェンスのプレイブック**（長く、しかも呼び出しのたびにまったく同じ内容です）、アソシエイトが手作業で用意した**正解ラベル**、そして採点者です。

そのうえで、引き継いだエージェントを、デモチームが書いたそのままの形で実行します。先を読む前に実行してください。違和感を嗅ぎ取るのも仕事のうちです。

In [ ]:
# ── 契約サンプル（248K件の契約群（estate）のラボ規模の代役）──────────
CONTRACTS = [
    {
        "id": "C-101", "vendor": "NorthWind Logistics GmbH",
        "text": """業務委託基本契約書：NorthWind Logistics GmbH（以下「サプライヤー」）とVolta Industrial Group（以下「顧客」）。

1. サービス。サプライヤーは、添付のSOWに従い、貨物輸送手配および倉庫管理サービスを提供する。
2. 期間および更新。当初期間は24か月とする。本契約は、いずれかの当事者が当該時点の期間満了の60日前までに更新拒絶の書面通知を行わない限り、12か月ずつ自動的に更新される。
3. 料金および支払。料金は別紙Aのとおりとする。請求書は請求日から45日以内に支払うものとし、支払遅延額には月1%の利息が発生する。
4. 責任の制限。秘密保持義務違反の場合を除き、本契約に起因する各当事者の責任の総額は、25万米ドル（$250,000）を超えないものとする。
5. 譲渡。いずれの当事者も、書面で通知することにより、相手方の同意なく、本契約を関連会社に譲渡し、または合併もしくは事業のほぼ全部の売却に伴って譲渡することができる。
6. 秘密保持。契約終了後3年間、標準的な相互の義務を負う。
7. 準拠法。本契約は、抵触法の原則にかかわらず、デラウェア州（Delaware）の法律に準拠する。""",
    },
    {
        "id": "C-102", "vendor": "Apex Facilities Co.",
        "text": """施設保守契約書：Apex Facilities Co.（以下「請負人」）とVolta Industrial Group（以下「発注者」）。

1. 範囲。別紙1に記載の拠点について、空調、電気設備、清掃の保守を行う。
2. 期間。発効日から36か月の固定期間とする。本契約は期間満了をもって終了し、自動更新されない。延長には、両当事者が署名した書面による変更契約が必要である。
3. 料金。月額固定料金$18,400に、材料費（原価に10%を上乗せ）を加える。
4. 責任の制限。請負人の本契約に基づく累計の責任総額は、10万米ドル（$100,000）を超えないものとする。いずれの当事者も、間接損害および結果的損害について責任を負わない。
5. 譲渡。いずれの当事者も、支配権の変更に伴う場合を含め、同意なく自由に本契約を譲渡することができる。
6. 保険。請負人は、1事故あたり$2,000,000の商業一般賠償責任保険を維持する。
7. 準拠法。本契約は、ニューヨーク州（New York）の法律に準拠する。""",
    },
    {
        "id": "C-103", "vendor": "Cobalt Data Services Ltd.",
        "text": """データ処理・ホスティング契約書：Cobalt Data Services Ltd.（以下「プロバイダー」）とVolta Industrial Group（以下「会社」）。

1. サービス。プロバイダーは、会社の工場テレメトリプラットフォームをホスティングし、運用データを処理する。
2. 期間および更新。当初期間は12か月とし、いずれかの当事者が90日前までに書面で解約通知を行わない限り、1年ずつ自動的に更新される。
3. サービスレベル。月間稼働率99.9%。サービスクレジットは別紙2のとおりとする。
4. 責任の制限。プロバイダーの責任の総額は、50万米ドル（$500,000）または過去12か月間に支払われた料金のいずれか大きい方を超えないものとする。ただし、上限はいかなる場合も$500,000を超えないものとする。
5. 譲渡および支配権の変更。いずれの当事者も、法の作用、支配権の変更、合併その他の方法を問わず、相手方の事前の書面による同意なく本契約を譲渡してはならない。ただし、当該同意は不合理に留保されないものとする。
6. データ保護。プロバイダーは、別紙CのDPAに従って個人データを処理する。
7. 準拠法。本契約は、カリフォルニア州（California）の法律に準拠する。""",
    },
    {
        "id": "C-104", "vendor": "Ironclad Security Services plc",
        "text": """警備サービス契約書：Ironclad Security Services plc（以下「Ironclad」）とVolta Industrial Group（以下「プリンシパル」）。

1. サービス。有人警備、警報対応、入退室管理を、プリンシパルの英国拠点に対して提供する。
2. 期間。開始日から12か月とし、満了により自動的に終了する。更新は、両当事者の書面による合意がある場合に限る。
3. 補償。Ironcladは、Ironcladの過失に起因する第三者の請求についてプリンシパルを補償する。プリンシパルは、現場調査で開示されなかった現場状況に起因する請求についてIroncladを補償する。各補償は上限がなく、終了後も6年間存続する。
4. 責任。両当事者は、第3条の補償を確認する。本契約は、それ以外には、いずれの当事者の責任についても、上限、最高額その他の制限を定めていない。
5. 譲渡および支配権の変更。Ironcladは、プリンシパルの支配権に変更があった場合、30日前の通知により本契約を解除することができる。いずれの当事者による譲渡も、相手方の事前の書面による同意を要する。
6. TUPE。両当事者は、期間満了時に警備要員へTUPE規則が適用される可能性があることを確認する。
7. 準拠法。本契約およびこれに起因する紛争は、イングランドおよびウェールズ（England and Wales）の法律に準拠する。""",
    },
    {
        "id": "C-105", "vendor": "Helios Components S.A.",
        "text": """供給契約書：Helios Components S.A.（以下「Helios」）とVolta Industrial Group（以下「買主」）。

1. 供給。Heliosは、買主の発注書に従い、別紙1に記載の精密部品を供給する。
2. 期間および更新。当初期間は24か月とし、いずれかの当事者が120日前までに通知しない限り、12か月ずつ自動的に延長される。
3. 価格。別紙2の価格表に従い、PPI指数に基づいて毎年調整する。上限は年4%とする。
4. 責任の制限。第9条（IP補償）を条件として、各当事者の本契約に基づく責任の総額は、100万米ドル（$1,000,000）を超えないものとする。
5. 譲渡および支配権の変更。買主の直接または間接の支配権の変更には、Heliosの事前の書面による同意を要する。Heliosは、自らの裁量により同意を留保することができる。
6. 品質。部品は別紙3の仕様に適合するものとし、不適合のロットはHeliosの費用で交換する。
7. 準拠法。本契約は、テキサス州（Texas）の法律に準拠する。

変更契約第2号（締結済みかつ発効済み）。両当事者は次のとおり合意する。本契約第4条（責任の制限）は全文を削除し、以後効力を有しない。疑義を避けるために付言すると、本変更契約の後、本契約はいずれの当事者の責任についても上限を定めていない。その他の条項は変更しない。""",
    },
    {
        "id": "C-106", "vendor": "Brightline Staffing LLC",
        "text": """人材サービス契約書：Brightline Staffing LLC（以下「Brightline」）とVolta Industrial Group（以下「顧客」）。

1. サービス。Brightlineは、要請に応じて、生産および倉庫業務の派遣人員を提供する。
2. 期間および更新。当初期間は1年とし、いずれかの当事者が更新の30日前までに書面で通知しない限り、1年ずつ自動的に更新される。
3. 料金。請求料金はRate Card v7のとおりとする。時間外は1.5倍。12か月以内に直接雇用する場合の紹介手数料は、初年度給与の20%とする。
4. 労働者の分類。Brightlineは雇用主として、賃金、源泉徴収、労働者災害補償保険に責任を負う。
5. 譲渡。いずれの当事者も、支配権の変更に伴う場合を含め、譲受人がすべての義務を引き受けることを条件に、相手方の同意なく本契約を譲渡することができる。
6. 保証。サービスは、専門的かつ熟練した方法で提供される。両当事者は、本契約に基づく責任について、いかなる制限または上限も合意していない。
7. 準拠法。本契約は、イリノイ州（Illinois）の法律に準拠する。""",
    },
]

# gold labels（正解ラベル）：実際の案件でアソシエイトがやるように、手作業で作成しています。
# risk_tierのルール（プレイブックにも記載）：HIGH = 支配権変更の同意が必要、かつ
# 賠償責任の上限なし。MEDIUM = このレッドフラグのどちらか一方のみ。LOW = どちらもなし。
GOLD = {
    "C-101": {"auto_renewal": True,  "change_of_control": False, "liability_cap_usd": 250000,
              "governing_law": "Delaware",          "risk_tier": "LOW"},
    "C-102": {"auto_renewal": False, "change_of_control": False, "liability_cap_usd": 100000,
              "governing_law": "New York",          "risk_tier": "LOW"},
    "C-103": {"auto_renewal": True,  "change_of_control": True,  "liability_cap_usd": 500000,
              "governing_law": "California",        "risk_tier": "MEDIUM"},
    "C-104": {"auto_renewal": False, "change_of_control": True,  "liability_cap_usd": None,
              "governing_law": "England and Wales", "risk_tier": "HIGH"},
    "C-105": {"auto_renewal": True,  "change_of_control": True,  "liability_cap_usd": None,
              "governing_law": "Texas",             "risk_tier": "HIGH"},
    "C-106": {"auto_renewal": True,  "change_of_control": False, "liability_cap_usd": None,
              "governing_law": "Illinois",          "risk_tier": "MEDIUM"},
}

print(f"Loaded {len(CONTRACTS)} contracts with gold labels")

**デューデリジェンスのプレイブック。** 人間でもモデルでも、すべてのレビュアーが同じ基準書に従って作業します。フィールドの定義、リスクの判定基準、条項ライブラリ、交渉の先例です。長く、権威があり、しかも**呼び出しのたびに1バイトも変わりません**。この言葉を覚えておいてください。

In [ ]:
PLAYBOOK_CORE = """You are ClauseScan, the contract-diligence reviewer for Project HELVETICA \
(Aldgate Capital Partners' acquisition of Volta Industrial Group). You review supplier \
contracts and report five audited fields. Work only from the contract text provided. Never \
guess: if the contract is silent on a point, report it as absent rather than inventing terms.

## Audited fields
1. auto_renewal (boolean) — true only if the contract renews automatically absent notice. A \
fixed term that expires, or renewal "by mutual written agreement", is NOT auto-renewal.
2. change_of_control (boolean) — true if assignment or change of control of either party \
requires the counterparty's prior consent, or gives the counterparty a termination right. \
Free assignment or notice-only assignment is false.
3. liability_cap_usd (number or null) — the aggregate liability cap in USD. null if no cap is \
stated, if liability is expressly uncapped, or if an amendment removed the cap. Amendments \
override the original clause — always check for amendments before reporting.
4. governing_law (string) — the governing jurisdiction, e.g. "Delaware" or "England and Wales".
5. risk_tier (LOW | MEDIUM | HIGH) — apply this rule exactly:
   HIGH = change-of-control consent/termination right present AND no liability cap.
   MEDIUM = exactly one of those two red flags present.
   LOW = neither red flag (no change-of-control restriction AND a cap is stated).

## Review discipline
- Read every section, then re-check the two red-flag clauses before concluding.
- Quote-check: the evidence you cite must appear in the contract.
- Deal context: Aldgate's purchase will itself trigger change-of-control clauses across the \
estate — that is why the field is audited.
"""

# 条項ライブラリ（clause library）と先例メモが、プレイブックを現実的な長さにしています。実際の案件では、
# これは貴社が維持している40ページの基準書PDFに相当します。
CLAUSE_LIBRARY = [
    ("Automatic renewal (evergreen)", "Term continues for successive periods unless a party gives notice by a stated deadline. Capture the notice window and the renewal period length.", "Notice windows under 30 days are operationally dangerous during an integration."),
    ("Fixed term with mutual-agreement renewal", "Agreement expires at the end of the stated term; any continuation needs a signed amendment.", "Often misread as auto-renewal. It is not."),
    ("Assignment with consent", "Neither party may assign without prior written consent; usually includes 'by operation of law' and merger language.", "This is a change-of-control restriction even when the words 'change of control' never appear."),
    ("Free assignment", "Either party may assign to affiliates or acquirers, sometimes with notice only.", "Notice-only assignment is NOT a change-of-control restriction."),
    ("Change-of-control termination right", "Counterparty may terminate upon a change of control of the other party.", "A termination right is as dangerous to deal value as a consent right — treat it as change_of_control = true."),
    ("Aggregate liability cap", "A single dollar ceiling on total liability under the agreement.", "Check whether carve-outs (confidentiality, IP, indemnity) sit outside the cap."),
    ("Capped at fees paid", "Liability limited to fees paid over a trailing window, sometimes with a dollar ceiling on top.", "Report the effective ceiling where one is stated; otherwise treat as a cap of unstated amount."),
    ("Uncapped liability", "No limitation-of-liability clause, or an express statement that liability is unlimited.", "Silence is a red flag, not a default. Report null."),
    ("Amendment overriding liability terms", "Later amendments may delete or rewrite the limitation-of-liability section.", "The amendment controls. Re-read amendments before reporting any cap."),
    ("Mutual indemnity, uncapped", "Each party indemnifies the other for defined claim classes with no ceiling.", "Uncapped indemnities alongside a deleted cap compound exposure."),
    ("Governing law", "The jurisdiction whose law governs the contract.", "Distinguish governing law from venue/forum; report the law."),
    ("Service levels and credits", "Uptime or performance commitments with credit remedies.", "Credits are usually the exclusive remedy — note but do not audit."),
    ("Payment terms", "Net-day windows, late interest, price-adjustment indices.", "Index-linked price escalators matter for the operating model, not this audit."),
    ("Confidentiality", "Mutual non-disclosure obligations with survival periods.", "Often carved out of the liability cap."),
    ("Insurance requirements", "Required coverage types and limits.", "Insurance limits are not liability caps. Do not conflate."),
    ("TUPE / employee transfer", "EU/UK staff-transfer regulations on outsourcing changes.", "Integration planning input; not an audited field."),
    ("Exclusivity", "Sole-supplier or minimum-purchase commitments.", "Flag in evidence notes if material; not an audited field."),
    ("Termination for convenience", "Either party may exit on notice without cause.", "Shortens effective exposure; note the notice window."),
    ("Most-favored-customer pricing", "Supplier promises pricing no worse than comparable customers.", "Diligence interest for procurement, not this audit."),
    ("IP indemnity", "Supplier defends infringement claims; sometimes uncapped.", "Carve-outs can sit outside the cap — read the cap clause's 'subject to' language."),
    ("Force majeure", "Excused performance on defined events.", "Not audited."),
    ("Audit rights", "Customer may audit supplier records or facilities.", "Not audited here; useful for integration."),
]

PRECEDENT_NOTES = [
    "Volta's 2024 acquisition of Kessler Tooling stalled five weeks on unflagged change-of-control consents across 1,100 supplier contracts.",
    "Aldgate's deal team treats any uncapped-liability supplier with consent rights as a Day-1 escalation.",
    "Notice windows: the integration office needs 90+ days of runway; sub-30-day windows go on the watch list.",
    "Caps stated in non-USD currencies are converted at the deal-model rate; report the stated figure and flag the currency in evidence.",
    "Where an MSA and SOW conflict, the MSA controls unless the SOW says otherwise — playbook v7, §3.2.",
    "Indemnity carve-outs outside the cap do not change liability_cap_usd; they belong in evidence.",
    "A termination-on-change-of-control right is reported as change_of_control = true even absent a consent requirement.",
    "Renewal 'by mutual written agreement' is not auto-renewal; the deal model treats those contracts as expiring.",
    "If two clauses conflict and no amendment resolves them, report the more conservative reading and say so in evidence.",
    "Evidence must cite section numbers; reviewers spot-check 10% of output against source text.",
]

WORKED_EXAMPLES = """
## Worked examples
Example A: a contract with free assignment and a $400,000 aggregate cap → change_of_control \
false, liability_cap_usd 400000, risk_tier LOW.
Example B: a contract requiring consent for change of control with a $1M cap → \
change_of_control true, cap 1000000, exactly one red flag → risk_tier MEDIUM.
Example C: a contract with a consent requirement whose cap was deleted by amendment → \
change_of_control true, liability_cap_usd null, both red flags → risk_tier HIGH.
"""


def _build_playbook() -> str:
    parts = [PLAYBOOK_CORE, "\n## Clause library\n"]
    for name, definition, red_flag in CLAUSE_LIBRARY:
        parts.append(f"### {name}\n{definition}\nRed flag note: {red_flag}\n")
    parts.append("\n## Negotiation precedent notes\n")
    for note in PRECEDENT_NOTES:
        parts.append(f"- {note}")
    parts.append(WORKED_EXAMPLES)
    text = "\n".join(parts)
    # 決定論的にパディングし、最小キャッシュ可能プレフィックスの最大値（Opus/Haikuで4096トークン、
    # 1トークンあたり約4文字と想定）を超えるようにして、どのモデルでもキャッシング（caching）の調整項目が働くようにする。
    annex = "\n\n## Annex: clause library (deal-team annotated re-issue)\n" + "\n".join(
        f"### {n}\n{d}\nRed flag note: {r}\n" for n, d, r in CLAUSE_LIBRARY)
    while len(text) < 22000:
        text += annex
    return text


PLAYBOOK = _build_playbook()
print(f"Playbook built: {len(PLAYBOOK):,} chars (~{len(PLAYBOOK)//4:,} tokens) — identical on every call")

**採点者。** 契約1件につき5つの監査対象フィールドを、決定論的なチェックと、手作業で用意した正解ラベルで採点します。精度は**ゲート**です。精度を壊す最適化は最適化ではありません。責任を自社の売上原価（COGS）の項目から、職業賠償責任保険へ付け替えているだけです。

In [ ]:
def _normalize_cap(value):
    """liability_cap_usd のモデル出力を float または None に変換します。"""
    if value is None:
        return None
    if isinstance(value, str):
        v = value.strip().lower().replace("$", "").replace(",", "")
        if v in ("", "none", "null", "n/a", "uncapped", "no cap"):
            return None
        try:
            return float(v)
        except ValueError:
            return None
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def grade_fields(fields: dict, gold: dict) -> dict:
    """1件の契約について、フィールドごとの合否を返します。"""
    f = fields or {}
    checks = {}
    checks["auto_renewal"] = bool(f.get("auto_renewal")) == gold["auto_renewal"]
    checks["change_of_control"] = bool(f.get("change_of_control")) == gold["change_of_control"]
    cap = _normalize_cap(f.get("liability_cap_usd"))
    if gold["liability_cap_usd"] is None:
        checks["liability_cap_usd"] = cap is None
    else:
        checks["liability_cap_usd"] = cap is not None and abs(cap - gold["liability_cap_usd"]) < 1
    gl, gg = str(f.get("governing_law") or "").lower(), gold["governing_law"].lower()
    checks["governing_law"] = bool(gl) and (gg in gl or gl in gg)
    checks["risk_tier"] = str(f.get("risk_tier") or "").strip().upper() == gold["risk_tier"]
    return checks


def extract_json(text: str):
    """自由記述テキストから、最初の有効なJSONオブジェクトを取り出します（v0では必要ですが、v1では不要かもしれません）。"""
    try:
        return json.loads(text.strip())
    except json.JSONDecodeError:
        pass
    for start, ch in enumerate(text):
        if ch != "{":
            continue
        depth = 0
        for end in range(start, len(text)):
            if text[end] == "{":
                depth += 1
            elif text[end] == "}":
                depth -= 1
                if depth == 0:
                    try:
                        return json.loads(text[start:end + 1])
                    except json.JSONDecodeError:
                        break
    return None


print("採点者の準備完了：監査対象フィールド5つ × 契約6件 = 30チェック、ゲートは90%以上（27/30）")

**ClauseScan v0：引き継いだままの姿。** 立て直し案件の初日にお客様のコードベースを読むつもりで読んでください。あざ笑うためではなく、お金と秒数がどこに消えているかを見つけるためです。

In [ ]:
EXTRACT_INSTRUCTION = (
    "以下の契約書について、5つの監査対象フィールドを判定してください。JSONオブジェクトで回答し、"
    "キーは auto_renewal, change_of_control, liability_cap_usd, governing_law, risk_tier, "
    "evidence です。\n\n契約書：\n"
)


def clausescan_v0(contract: dict) -> dict:
    """デモチームが出荷したパイプライン。2パス、すべてOpus、キャッシュなし、
    冗長な出力、ストリーミングなし、構造上シーケンシャル。"""
    t0 = time.perf_counter()
    calls = []

    # PASS 1：「まず契約をしっかり理解する」（誰も読まない詳細なブリーフィング）
    r1 = client.messages.create(
        model=MODEL_OPUS, max_tokens=8000,
        system=PLAYBOOK,  # 5K+ tokens を、呼び出しのたびに定価で再課金
        messages=[{"role": "user", "content":
                   "抽出に着手する前に、この契約書について条項ごとの詳細なブリーフィングを書いてください。"
                   "気になる点にはコメントを添えてください。\n\n"
                   + contract["text"]}],
    )
    calls.append((MODEL_OPUS, r1.usage))

    # PASS 2：抽出。ブリーフィングと契約書に加えて、プレイブックをもう一度渡す
    r2 = client.messages.create(
        model=MODEL_OPUS, max_tokens=8000,
        system=PLAYBOOK,
        messages=[{"role": "user", "content":
                   "次は、ある契約書の社内ブリーフィングです。\n\n" + text_of(r1)
                   + "\n\n次に、推論の過程を段階的に詳しく説明したうえで、JSONオブジェクトを出力してください。"
                   + "キーは auto_renewal, change_of_control, "
                   + "liability_cap_usd, governing_law, risk_tier, evidence です。\n\n契約書：\n"
                   + contract["text"]}],
    )
    calls.append((MODEL_OPUS, r2.usage))

    return {
        "fields": extract_json(text_of(r2)),
        "calls": calls,
        "elapsed": time.perf_counter() - t0,
    }


print("ClauseScan v0を読み込みました。次のセルでポートフォリオを実行します")

In [ ]:
def run_portfolio(pipeline, contracts=CONTRACTS, gold=GOLD, workers: int = 0,
                  warm_first: bool = True) -> dict:
    """契約サンプルに対してパイプラインを実行し、契約ごとに採点、計時、コスト計算を行います。"""

    def run_one(contract):
        out = pipeline(contract)
        checks = grade_fields(out["fields"], gold[contract["id"]])
        return {
            "id": contract["id"], "vendor": contract["vendor"],
            "fields": out["fields"], "checks": checks,
            "n_correct": sum(checks.values()), "n_fields": len(checks),
            "elapsed": out["elapsed"],
            "cost": sum(calculate_cost(m, u) for m, u in out["calls"]),
            "calls": len(out["calls"]),
        }

    wall0 = time.perf_counter()
    if workers and len(contracts) > 1:
        rows = []
        head = contracts[0]
        if warm_first:
            rows.append(run_one(head))  # ファンアウト（fan-out）の前に、1回の呼び出しでキャッシュを書き込む
            remaining = contracts[1:]
        else:
            remaining = contracts
        with ThreadPoolExecutor(max_workers=workers) as ex:
            rows.extend(ex.map(run_one, remaining))
    else:
        rows = [run_one(c) for c in contracts]
    wall = time.perf_counter() - wall0

    total_fields = sum(r["n_fields"] for r in rows)
    return {
        "rows": rows,
        "accuracy": sum(r["n_correct"] for r in rows) / total_fields,
        "p50_s": statistics.median(r["elapsed"] for r in rows),
        "cost_per_contract": sum(r["cost"] for r in rows) / len(rows),
        "total_cost": sum(r["cost"] for r in rows),
        "wall_s": wall,
    }


def print_report(report: dict, label: str):
    rows = [[r["id"], r["vendor"][:28], f"{r['n_correct']}/{r['n_fields']}",
             f"{r['elapsed']:.1f}s", f"${r['cost']:.4f}", r["calls"]]
            for r in report["rows"]]
    print(f"\n── {label} " + "─" * max(1, 64 - len(label)))
    print(tabulate(rows, headers=["ID", "Vendor", "Fields", "TTC", "Cost", "Calls"],
                   tablefmt="simple"))
    print(f"\n  accuracy {report['accuracy']*100:.0f}%   ·   p50 {report['p50_s']:.1f}s/contract"
          f"   ·   ${report['cost_per_contract']:.4f}/contract"
          f"   ·   batch wall-clock {report['wall_s']:.0f}s")


print("契約6件のサンプルでClauseScan v0を実行中（逐次実行、約2〜4分。"
      "遅さこそがデータです）...")
BASELINE = run_portfolio(clausescan_v0)
print_report(BASELINE, "ClauseScan v0：引き継いだ基準値")

sla_p50 = "PASS" if BASELINE["p50_s"] <= 5 else "FAIL"
print(f"\n  SLA check: p50 ≤ 5s → {sla_p50}   ·   accuracy ≥ 90% → "
      f"{'PASS' if BASELINE['accuracy'] >= 0.9 else 'FAIL'}")

**診断。** v0は*正しく動きます*。それこそが危険な理由です。デモでは何も壊れて見えないので出荷され、そのムダは248,000倍に積み上がります。罪状を挙げましょう。

1. **何でもOpus。** 契約群の大半は定型文で、もっと安いモデルでも十分に読めます。
2. **5Kトークンのプレイブックを、呼び出しのたびに定価で再課金している**：契約1件につき2回です。中身は一切変わりません。プロンプトキャッシングの教科書的な候補です。
3. **1回で済むはずの直列往復が2回。** 「ブリーフィング」パスはレイテンシを倍にし、自分の出力を次の呼び出しの入力として課金します。
4. **冗長な自由記述の出力：** 「段階的に詳しく説明してから、JSON」。出力トークンは入力トークンの**5倍の価格**で、余計な1トークンごとに実時間もかかります。
5. **ストリーミングなし。** アナリストはTTCの間ずっと、固まった画面を見つめます。
6. **逐次バッチ。** 6件の契約は6件分の実時間がかかり、248,000件なら25万件分かかります。

このうち5つは、残った呼び出しを安く、あるいは速くするために引く調整項目です。6つ目の往復は違います。ほかを最適化する前に、呼び出しごと削除するものです。パート4で、メーターを回しながら1つずつ見ていきます。

# パート4 · 6つの調整項目

**まず、不要だった往復を排除します。** 個々の呼び出しを安くする前に、そもそも不要だった呼び出しを削除してください。直列の往復は、そのたびにフルコストを払います。ネットワーク、プリフィル、生成に加えて、連鎖した呼び出しは前の出力を新しい入力として再課金します。v0の「ブリーフィング」パスは、まるごと往復の税金です。同じ税金は、ツール呼び出し（リクエスト → `tool_use` → 実行 → 結果 → 応答）やマルチステップのチェーンでも、再び支払うことになります。レイテンシに見合わないステップは統合しましょう。v0の2パスは1パスになり、以下のすべての調整項目は、この残った1回の呼び出しに対して働きます。

## 調整項目1 · プロンプトキャッシング：プレイブックを買い直さない

これで契約1件につき呼び出しは1回です。プレイブックの買い直しをやめさせましょう。

キャッシングは**プレフィックスマッチ**です。tools → system → messages の順で、1バイトでも変わると、それ以降はすべて無効になります。このプレイブックは、設計上、固定されたプレフィックスです。`cache_control`を付けると、APIは処理済みのプレフィックスを保存します。最初の呼び出しは**1.25倍**の書き込み割増を払い、TTL（5分）内のすべての呼び出しは**0.1倍**で読み出します。しかも再処理が不要になるので、TTFTも短くなります。

お客様案件での現場メモ：キャッシュ可能なプレフィックスの最小サイズは、**Opus/Haikuで4096トークン、Sonnetで2048トークン**です（それより短いプレフィックスは、警告なしにキャッシュされません）。タイムスタンプ、リクエストID、契約書そのものなど、変動する内容はキャッシュするブロックの*後ろ*に置きます。確認は雰囲気ではなく、`usage.cache_read_input_tokens`で行います。

In [ ]:
CACHED_SYSTEM = [{"type": "text", "text": PLAYBOOK, "cache_control": {"type": "ephemeral"}}]


def cached_extract(contract, model=MODEL_SONNET):
    t0 = time.perf_counter()
    resp = client.messages.create(
        model=model, max_tokens=1200, system=CACHED_SYSTEM,
        messages=[{"role": "user", "content": EXTRACT_INSTRUCTION + contract["text"]}],
    )
    return resp, time.perf_counter() - t0


for label in ("コールド（キャッシュ書き込み）", "ウォーム（キャッシュ読み出し）"):
    resp, secs = cached_extract(CONTRACTS[0])
    u = resp.usage
    print(f"{label:20s} {secs*1000:6.0f}ms · uncached_in={u.input_tokens:>5} · "
          f"cache_write={u.cache_creation_input_tokens or 0:>5} · "
          f"cache_read={u.cache_read_input_tokens or 0:>5} · "
          f"cost=${calculate_cost(MODEL_SONNET, u):.5f}")

print("\ncache_readが0からプレイブック相当のサイズまで跳ね上がり、それに合わせてコストが下がることを確認してください。")

## 調整項目2 · モデルルーティング：モデルではなくポートフォリオ

248K件の契約群のうち、本当に上級者の目が必要なのは、おそらく15〜25%です。条件を書き換える変更契約、矛盾する条項、異例の補償などです。残りは定型文です。そこで、**Haikuでトリアージのパス**（安く、速く、出力は極小）を実行し、振り分けます。ROUTINE → HaikuまたはSonnet、COMPLEX → SonnetまたはOpus。これは人員配置の発想で、皆さんの会社がすでによく理解していることです。すべてのNDAにシニアパートナーを付けたりはしません。

In [ ]:
TRIAGE_SYSTEM = (
    "あなたは、デューデリジェンスのパイプラインでサプライヤー契約をトリアージします。契約が次のいずれかに該当する場合は "
    "COMPLEX に分類してください。該当するのは、以前の条件を変更する変更契約、互いに矛盾する条項、"
    "異例の補償または上限のない補償、責任条項が明記されておらず、曖昧または暗黙にとどまる場合です。"
    "それ以外は ROUTINE に分類してください。"
)

TRIAGE_SCHEMA = {
    "type": "json_schema",
    "schema": {
        "type": "object",
        "properties": {
            "complexity": {"type": "string", "enum": ["ROUTINE", "COMPLEX"]},
            "reason": {"type": "string", "description": "短い1文で。"},
        },
        "required": ["complexity", "reason"],
        "additionalProperties": False,
    },
}


def triage(contract):
    resp = client.messages.create(
        model=MODEL_HAIKU, max_tokens=150, system=TRIAGE_SYSTEM,
        messages=[{"role": "user", "content": contract["text"]}],
        output_config={"format": TRIAGE_SCHEMA},
    )
    verdict = extract_json(text_of(resp)) or {}
    return verdict.get("complexity", "COMPLEX"), resp  # フェイルセーフ：不明なら COMPLEX


rows = []
for c in CONTRACTS:
    verdict, resp = triage(c)
    rows.append([c["id"], c["vendor"][:28], verdict,
                 f"${calculate_cost(MODEL_HAIKU, resp.usage):.5f}"])
print(tabulate(rows, headers=["ID", "Vendor", "Triage", "トリアージのコスト"], tablefmt="simple"))
print("\nルーティングの判断は、1セントの数分の1で買えます。節約は、それによって"
      "Opusから振り分けられた分から生まれます。")

## 調整項目3 · 出力の規律：エッセイではなくスキーマ

出力トークンは入力トークンの**5倍**のコストがかかり、1トークンごとに生成時間もかかります（TTC ≈ TTFT + output_tokens ÷ OTPS）。v0は、ステップごとの文章*に加えて*JSONを求めます。解決策は**構造化出力**、つまり`output_config.format`とJSONスキーマです。応答そのものが成果物になります。有効なJSONで、パース用の正規表現も前置きも要りません。リスクの判定基準はフィールドの説明に書かれているので、スキーマはQA担当が承認する仕様書も兼ねます。

同じ近所にある2つのダイヤル：
- `max_tokens`：厳格な上限です。成果物に合わせて適正化します（このJSONに必要なのは約300で、8000ではありません）。
- `output_config.effort`：Sonnet/Opusは`low`〜`max`を受け付け、深さとトークン数を引き換えにできます。**Haiku 4.5ではエラーになります。** Haikuはすでに低レイテンシ層だからです。

In [ ]:
EXTRACTION_SCHEMA = {
    "type": "json_schema",
    "schema": {
        "type": "object",
        "properties": {
            "auto_renewal": {"type": "boolean",
                             "description": "通知がなければ契約が自動的に更新される場合にのみTrue。"},
            "change_of_control": {"type": "boolean",
                                  "description": "譲渡・支配権の変更に同意が必要、または解除権が生じる場合にTrue。"},
            "liability_cap_usd": {"anyOf": [{"type": "number"}, {"type": "null"}],
                                  "description": "USDでの賠償責任の総額上限。上限の記載がない場合、または変更契約で削除された場合はnull。"},
            "governing_law": {"type": "string",
                              "description": "法域（英語表記）。例：'Delaware'、'England and Wales'。"},
            "risk_tier": {"type": "string", "enum": ["LOW", "MEDIUM", "HIGH"],
                          "description": "HIGH = 支配権変更の制限あり、かつ上限なし。MEDIUM = レッドフラグがちょうど1つ。LOW = どちらもなし。"},
            "evidence": {"type": "string",
                         "description": "根拠にした条番号を示す1文。"},
        },
        "required": ["auto_renewal", "change_of_control", "liability_cap_usd",
                     "governing_law", "risk_tier", "evidence"],
        "additionalProperties": False,
    },
}


def extract_structured(contract, model=MODEL_SONNET, cached=True, effort=None):
    """1回の呼び出し、スキーマで制約した出力、キャッシュ済みプレイブックとエフォート調整は任意。"""
    t0 = time.perf_counter()
    output_config = {"format": EXTRACTION_SCHEMA}
    if effort and model != MODEL_HAIKU:   # effortはHaiku 4.5ではエラーになる
        output_config["effort"] = effort
    resp = client.messages.create(
        model=model, max_tokens=1000,
        system=CACHED_SYSTEM if cached else PLAYBOOK,
        messages=[{"role": "user", "content": EXTRACT_INSTRUCTION + contract["text"]}],
        output_config=output_config,
    )
    return {"fields": extract_json(text_of(resp)), "calls": [(model, resp.usage)],
            "elapsed": time.perf_counter() - t0}


demo = extract_structured(CONTRACTS[0], model=MODEL_SONNET, effort="low")
model_used, u = demo["calls"][0]
v0_row = BASELINE["rows"][0]
print(f"Structured single pass: {demo['elapsed']:.1f}s · {u.output_tokens} output tokens · "
      f"${calculate_cost(model_used, u):.5f}")
print(f"v0 on the same contract: {v0_row['elapsed']:.1f}s · ${v0_row['cost']:.4f} across "
      f"{v0_row['calls']} calls — most of it prose nobody reads.")
print(json.dumps(demo["fields"], indent=2))

## 調整項目4 · ストリーミング：TTFTはUXの数字

残った1回の呼び出しをストリーミングします。TTCは経済性の数字ですが、**TTFTはUXの数字**です。答えが組み上がるのを見守るアナリストと、メールに逃げるアナリストを分ける差になります。

In [ ]:
t0 = time.perf_counter()
ttft, total, sresp = _stream_request(
    [{"role": "user", "content": EXTRACT_INSTRUCTION + CONTRACTS[0]["text"]}],
    model=MODEL_SONNET, max_tokens=1000, system=CACHED_SYSTEM,
    output_config={"format": EXTRACTION_SCHEMA},
)
print(f"Streamed single pass:  TTFT {ttft*1000:.0f}ms · TTC {total*1000:.0f}ms")
print(f"v0 two-pass TTC on this contract was {BASELINE['rows'][0]['elapsed']:.1f}s — "
      f"and its TTFT *was* its TTC, because nothing streamed.")

## 調整項目5 · ポートフォリオを並列化する（先にキャッシュをウォームする）

契約は互いに独立しているので、同時にレビューできます。プロが押さえる細かい点が1つあります。キャッシュエントリは、最初の応答が**ストリーミングを開始して**はじめて読み出せるようになります。同一のリクエストN件を同時に送ると、すべてコールド価格を払います。そこで、**まず契約を1件送ってキャッシュをウォームし、それから展開します。**（`run_portfolio`は、ここまでずっとこれを自動でやっています：`warm_first=True`）

現場メモ：レート制限のティアを守ってください。SDKは429をバックオフ付きで自動リトライしますが、TPM上限に合わせたファンアウトは、リトライで空回りするファンアウトより安く済みます。

In [ ]:
seq = run_portfolio(lambda c: extract_structured(c, model=MODEL_SONNET), workers=0)
par = run_portfolio(lambda c: extract_structured(c, model=MODEL_SONNET), workers=4)
print(f"Sequential portfolio wall-clock: {seq['wall_s']:.0f}s")
print(f"Parallel (warm-first, 4 workers): {par['wall_s']:.0f}s")
print(f"Accuracy held at {par['accuracy']*100:.0f}% — speed that costs accuracy is not speed.")

## 調整項目6 · 2速アーキテクチャ：Batch APIレーン

すべての契約にアナリストが張り付く必要はありません。案件チームのように、作業を2つに分けましょう。

- **インタラクティブレーン**：作業セッション、ストリーミング、p50 ≤ 5s。ここまでに作ってきたものすべてです。
- **バックフィルレーン**：残りの約235K件の文書は、**Batch API**で夜間に処理します。**すべてのトークン料金が50%オフ**です（ほとんどのバッチは1時間以内に完了し、結果は29日間保存されます）。

同じモデル、同じプロンプト、同じスキーマで、価格は半分です。その代わり、どうせ使う予定のなかったレイテンシを差し出します。この分割、つまり*お客様が触れるもの*と*裏で動くもの*の分離は、大量処理案件で最もレバレッジの大きいアーキテクチャ上の問いです。

In [ ]:
from anthropic.types.message_create_params import MessageCreateParamsNonStreaming
from anthropic.types.messages.batch_create_params import Request


def build_batch_requests(contracts):
    return [
        Request(
            custom_id=c["id"],
            params=MessageCreateParamsNonStreaming(
                model=MODEL_SONNET, max_tokens=1000,
                system=CACHED_SYSTEM,  # キャッシュ（cache）はバッチでも機能する（ヒットはベストエフォート）
                messages=[{"role": "user", "content": EXTRACT_INSTRUCTION + c["text"]}],
                output_config={"format": EXTRACTION_SCHEMA},
            ),
        )
        for c in contracts
    ]


batch_requests = build_batch_requests(CONTRACTS)
print(f"Built {len(batch_requests)} batch requests "
      f"(limits: 100K requests / 256 MB per batch)")

interactive_cpc = par["cost_per_contract"]
print(f"\nInteractive lane:  ${interactive_cpc:.4f}/contract")
print(f"Batch lane (50%):  ${interactive_cpc * BATCH_DISCOUNT:.4f}/contract")

RUN_BATCH = False  # 実際に送信するにはTrueに変更する。ポーリングは client.messages.batches.retrieve() で行う
if RUN_BATCH and PROVIDER == "bedrock":
    # Batch APIはAnthropic APIのエンドポイントであり、Amazon Bedrockでは提供されていません。
    # 上記はすべてそのまま成り立ちます。リクエスト（request）は作成済みで、以下の50%オフの経済性こそ
    # が本当の学びどころです。Bedrockには独自のバッチ推論サービスがあり、
    # APIは異なりますが、調整項目（急がない処理を後回しにして安く済ませる）は同じです。
    print("⚠️  実際の送信をスキップします：Batch APIはAmazon Bedrockでは利用できません。")
    print("    上で作成したバッチリクエストは有効です。内容を確認し、")
    print("    以下のコストモデルを読んでください。Bedrockでは独自のバッチ推論サービスを使います。")
elif RUN_BATCH:
    batch = client.messages.batches.create(requests=batch_requests)
    print(f"Submitted batch {batch.id} — status: {batch.processing_status}")
    print("ポーリング：client.messages.batches.retrieve(batch.id); "
          "結果：client.messages.batches.results(batch.id)")

# パート5 · 最適化スプリント 🏁

ここからはあなたの番です。下の`clausescan_v1`は、v0の忠実なコピーとして始まり、スコアは**約100**になります（これが基準値の指数です）。`CONFIG`を編集して調整項目を動かし、スコアカードを再実行してください。`CONFIG`だけでは足りなくなったら、関数そのものを編集してかまいません。むしろ推奨です。

**スコアリング**（SLAを1つの数字にまとめたもの）：

```
accuracy < 90%  →  SCORE = 0 (gate — no partial credit for fast, wrong answers)
otherwise       →  SCORE = 50 × (baseline_$ / your_$) + 50 × (baseline_p50 / your_p50)
```

v0 = 100。コストを4分の1、レイテンシを2分の1にすれば300です。強い構成は400を超えます。

**参加ルール：**
1. サンプルで精度90%以上。さらに、パイプラインは契約書を*読まなければなりません*。答えをハードコードすると、ホールドアウトセットでは0点になります（しかもコードは全員の前で読み上げられます）。
2. モデルも、キャッシュも、ルーティングも、並列度も自由です。APIの範囲内での創意工夫が主眼です。
3. 実行のたびに、リーダーボードの行をセッションのチャットに投稿してください。表を塗り替えましょう。

### ✏️ あなたの番：このCONFIGがハンドルです

ここを編集して再実行し、スコアの変化を見てください。

In [ ]:
# ✏️ あなたの番：このCONFIGを編集してください
CONFIG = {
    # 調整項目2：ルーティング。試してみる：triage_routing=True, routine_model=MODEL_HAIKU,
    #                         complex_model=MODEL_SONNET
    "triage_routing": False,
    "routine_model": MODEL_OPUS,
    "complex_model": MODEL_OPUS,

    # 調整項目1：プレイブックのプレフィックス（prefix）をキャッシュする
    "cache_playbook": False,

    # 調整項目3 + 往復の削減：ブリーフィング → 文章 → JSON ではなく、スキーマ付きの1パスにする
    "structured_single_pass": False,
    "max_tokens": 8000,           # 出力を絞ったら適正化（right-size）する（約1000）
    "effort": None,               # "low" | "medium" | "high"：Sonnet/Opusのみ

    # 調整項目5：ポートフォリオの並列実行（0 = 逐次、warm-firstは自動）
    "parallel_workers": 0,
}


def clausescan_v1(contract: dict) -> dict:
    """あなたのパイプライン。最初はv0で、どこまで育てるかは自分次第です。"""
    t0 = time.perf_counter()
    calls = []

    # ルーティング（routing、調整項目2）
    model = CONFIG["routine_model"]
    if CONFIG["triage_routing"]:
        verdict, tri_resp = triage(contract)
        calls.append((MODEL_HAIKU, tri_resp.usage))
        model = CONFIG["complex_model"] if verdict == "COMPLEX" else CONFIG["routine_model"]

    # キャッシュ済み（cached）プレイブックと生（raw）のプレイブック（調整項目1）
    system = CACHED_SYSTEM if CONFIG["cache_playbook"] else PLAYBOOK

    if CONFIG["structured_single_pass"]:
        # 調整項目3 + 4：1パス、スキーマ出力、任意のeffort調整
        output_config = {"format": EXTRACTION_SCHEMA}
        if CONFIG["effort"] and model != MODEL_HAIKU:
            output_config["effort"] = CONFIG["effort"]
        resp = client.messages.create(
            model=model, max_tokens=CONFIG["max_tokens"], system=system,
            messages=[{"role": "user", "content": EXTRACT_INSTRUCTION + contract["text"]}],
            output_config=output_config,
        )
        calls.append((model, resp.usage))
        fields = extract_json(text_of(resp))
    else:
        # v0の2パスのフロー（verbatim）をそのまま再現
        r1 = client.messages.create(
            model=model, max_tokens=CONFIG["max_tokens"], system=system,
            messages=[{"role": "user", "content":
                       "抽出に着手する前に、この契約書について条項ごとの詳細なブリーフィングを書いてください。"
                       "気になる点にはコメントを添えてください。\n\n"
                       + contract["text"]}],
        )
        calls.append((model, r1.usage))
        r2 = client.messages.create(
            model=model, max_tokens=CONFIG["max_tokens"], system=system,
            messages=[{"role": "user", "content":
                       "次は、ある契約書の社内ブリーフィングです。\n\n" + text_of(r1)
                       + "\n\n次に、推論の過程を段階的に詳しく説明したうえで、"
                       + "JSONオブジェクトを出力してください。キーは auto_renewal, change_of_control, "
                       + "liability_cap_usd, governing_law, risk_tier, evidence です。\n\n契約書：\n"
                       + contract["text"]}],
        )
        calls.append((model, r2.usage))
        fields = extract_json(text_of(r2))

    return {"fields": fields, "calls": calls, "elapsed": time.perf_counter() - t0}


def engagement_score(report: dict, baseline: dict) -> int:
    if report["accuracy"] < 0.90:
        return 0
    return round(50 * baseline["cost_per_contract"] / max(report["cost_per_contract"], 1e-9)
                 + 50 * baseline["p50_s"] / max(report["p50_s"], 1e-9))


def run_scorecard(pipeline, label="clausescan_v1", contracts=CONTRACTS, gold=GOLD):
    report = run_portfolio(pipeline, contracts=contracts, gold=gold,
                           workers=CONFIG.get("parallel_workers", 0))
    print_report(report, label)
    score = engagement_score(report, BASELINE)
    levers = []
    if CONFIG.get("triage_routing"):
        levers.append("routing")
    if CONFIG.get("cache_playbook"):
        levers.append("cache")
    if CONFIG.get("structured_single_pass"):
        levers.append("schema-1pass")
    if CONFIG.get("effort"):
        levers.append(f"effort-{CONFIG['effort']}")
    if CONFIG.get("parallel_workers"):
        levers.append(f"parallel-x{CONFIG['parallel_workers']}")
    gate = "" if report["accuracy"] >= 0.90 else "  ⛔ 精度ゲート不通過：スコアは0になります"
    print(f"\n  ENGAGEMENT SCORE: {score}  (v0 baseline = 100){gate}")
    print(f"  📋 leaderboard line →  SCORE {score} · acc {report['accuracy']*100:.0f}% · "
          f"p50 {report['p50_s']:.1f}s · ${report['cost_per_contract']:.4f}/contract · "
          f"levers: {'+'.join(levers) if levers else 'none'}")
    return report


print("スプリント用ハーネスの準備完了。上のCONFIGを編集して、次のセルを実行してください。納得するまで繰り返しましょう。")

In [ ]:
my_report = run_scorecard(clausescan_v1)

**イテレーション。** おすすめの進め方を示しますが、自分なりの道を探してください。リーダーボードは発想力を評価します。

1. `structured_single_pass=True, max_tokens=1000`：出力トークン数とTTCが激減するのを観察します。
2. `cache_playbook=True`：2回実行します。2回目で、ウォームキャッシュの経済性が見えます。
3. `triage_routing=True, routine_model=MODEL_HAIKU, complex_model=MODEL_SONNET`：ポートフォリオ戦略です。
4. `parallel_workers=4`：作業セッションの実時間を短縮します。
5. そのうえで、型から外れてみましょう。`EXTRACT_INSTRUCTION`を削る、Sonnetで`effort="low"`を試す、HIGHリスクの契約はセカンドオピニオンとしてOpusに回す、定型の契約ではプレイブックを外して複雑な契約でだけ残す（キャッシュヒットにどう影響するでしょう？）、セッション開始時に`max_tokens=0`のリクエストでキャッシュを事前にウォームする……

終わったと思ったら、ホールドアウトです。パイプラインがまだ見たことのない契約が2件あります。サンプルでしか動かない最適化は、私たちの業界では過学習と呼ばれ、お客様の側では指摘事項になるからです。

In [ ]:
HOLDOUT_CONTRACTS = [
    {
        "id": "C-201", "vendor": "Vanta Marketing Collective",
        "text": """マーケティングサービス契約書：Vanta Marketing Collective（以下「代理店」）とVolta Industrial Group（以下「顧客」）。

1. サービス。承認済みの作業指示書に基づく、ブランド、デジタル、展示会のマーケティングサービス。
2. 期間。発効日から18か月とし、期間満了をもって自動的に終了する。更新には、両当事者が締結する新たな契約が必要である。
3. 料金。月額リテーナー$22,000に、事前承認済みの実費を加える。
4. 責任の制限。本契約に基づく各当事者の責任の総額は、7万5,000米ドル（$75,000）に制限される。ただし、第5条（IP請求に対する補償）に基づき支払うべき金額のみを除く。
5. 補償。代理店は、代理店が作成した素材に起因する第三者のIP請求について、顧客を補償する。
6. 譲渡。いずれの当事者も、支配権の変更に伴う場合を含め、書面で通知することにより、同意なく本契約を譲渡することができる。
7. 準拠法。本契約は、カナダ、オンタリオ州（Ontario）の法律に準拠する。""",
    },
    {
        "id": "C-202", "vendor": "Quarry Industrial Supply Pte. Ltd.",
        "text": """工業用品供給契約書：Quarry Industrial Supply Pte. Ltd.（以下「Quarry」）とVolta Industrial Group（以下「購入者」）。

1. 供給。Quarryは、確定した発注書に従い、研磨材、締結部品、消耗品を供給する。
2. 期間および更新。当初期間は2年とし、いずれかの当事者が90日前までに更新拒絶の書面通知を行わない限り、1年ずつ自動的に更新される。
3. 価格。別紙Aの価格ファイルに従う。S$5,000を超える注文は、運賃前払いとする。
4. 責任の制限。本契約に起因または関連するQuarryおよび購入者それぞれの責任の総額は、合計200万米ドル（$2,000,000）を超えないものとする。
5. 譲渡および支配権の変更。いずれの当事者も、合併、買収、支配権の変更によるものを含め、相手方の事前の書面による同意なく、本契約を譲渡または移転してはならない。同意のない譲渡の試みは無効とする。
6. コンプライアンス。各当事者は、適用される輸出管理法および反腐敗法を遵守する。
7. 準拠法。本契約は、シンガポール（Singapore）の法律に準拠する。""",
    },
]

HOLDOUT_GOLD = {
    "C-201": {"auto_renewal": False, "change_of_control": False, "liability_cap_usd": 75000,
              "governing_law": "Ontario",   "risk_tier": "LOW"},
    "C-202": {"auto_renewal": True,  "change_of_control": True,  "liability_cap_usd": 2000000,
              "governing_law": "Singapore", "risk_tier": "MEDIUM"},
}

RUN_HOLDOUT = False  # CONFIGが確定したらTrueに変更する
if RUN_HOLDOUT:
    run_scorecard(clausescan_v1, label="clausescan_v1 — HOLDOUT",
                  contracts=HOLDOUT_CONTRACTS, gold=HOLDOUT_GOLD)
else:
    print("ホールドアウトの準備完了。CONFIGが確定したら RUN_HOLDOUT = True にしてください。"
          "ホールドアウトに合わせたチューニングは禁止です。それがルールです。")

# パート6 · ステアリングコミッティ向けスライド

ノートブックの中にとどまる最適化は、趣味にすぎません。このセルは、測定した数字を、ステアリングコミッティが実際に読む「変更前後の経済性」に変換します。前提も併せて表示します。前提のない効果試算は、お客様の監査委員会が真っ先に却下するものだからです。

In [ ]:
ASSUMPTIONS = {
    "contracts_in_estate": 248_000,
    "production_scale_factor": 8,   # 本番の契約（production）はラボのサンプルの約8倍のトークン数。入力が支配的なので、
                                    # コストはドキュメントの長さ（document length）にほぼ比例して増える
    "interactive_share": 0.15,      # アナリストの作業セッション（full price、ストリーミング）
    "batch_share": 0.85,            # 夜間のBatch APIによるバックフィル（50%オフ）
    "fee_per_contract": 0.75,       # HELVETICAがレビュー済み契約1件あたりに請求する金額
}


def steering_committee_slide(baseline: dict, optimized: dict, a=ASSUMPTIONS):
    n, scale = a["contracts_in_estate"], a["production_scale_factor"]

    def estate_cogs(cpc, two_speed: bool):
        per = cpc * scale
        if two_speed:
            per = per * a["interactive_share"] + per * a["batch_share"] * BATCH_DISCOUNT
        return per * n

    before = estate_cogs(baseline["cost_per_contract"], two_speed=False)
    after = estate_cogs(optimized["cost_per_contract"], two_speed=True)
    revenue = n * a["fee_per_contract"]

    rows = [
        ["精度（監査対象フィールド）", f"{baseline['accuracy']*100:.0f}%",
         f"{optimized['accuracy']*100:.0f}%", "ゲート維持"],
        ["p50レイテンシ", f"{baseline['p50_s']:.1f}s", f"{optimized['p50_s']:.1f}s",
         f"{baseline['p50_s']/max(optimized['p50_s'],1e-9):.1f}× faster"],
        ["ラボコスト / 契約あたり", f"${baseline['cost_per_contract']:.4f}",
         f"${optimized['cost_per_contract']:.4f}",
         f"{baseline['cost_per_contract']/max(optimized['cost_per_contract'],1e-9):.1f}× cheaper"],
        ["全体のCOGS（248K件の文書）", f"${before:,.0f}", f"${after:,.0f}",
         f"${before-after:,.0f} saved"],
        ["案件マージン", f"{(revenue-before)/revenue*100:.0f}%",
         f"{(revenue-after)/revenue*100:.0f}%", "契約あたり$0.75の料金ベース"],
        ["インタラクティブSLA（p50 ≤ 5s）",
         "PASS" if baseline["p50_s"] <= 5 else "FAIL",
         "PASS" if optimized["p50_s"] <= 5 else "FAIL", ""],
    ]
    print("PROJECT HELVETICA：推論の最適化、変更前と変更後")
    print(tabulate(rows, headers=["Metric", "v0 (inherited)", "Optimized", "Delta"],
                   tablefmt="grid"))
    print("\nAssumptions: " + ", ".join(f"{k}={v}" for k, v in a.items()))
    print("本番のCOGSは、ラボの契約あたり$ × スケール係数でモデル化。最適化後のレーン配分は "
          f"{a['interactive_share']:.0%} interactive / {a['batch_share']:.0%} batch at 50% off.")


steering_committee_slide(BASELINE, my_report)

# まとめ：お客様向けチェックリストとしての調整項目

推論コストの話し合いには、このリストを持って臨めば、報酬に見合う仕事ができます。

1. **まず測る**：TTFT、TTC、OTPS、キャッシュを考慮した単位あたり$。基準値がなければ、効果試算もありません。
2. **モデルポートフォリオ + ルーティング**：大量処理には安いモデル、判断には高いモデル、そしてどちらかを決めるトリアージのパス。
3. **固定プレフィックスをキャッシュする**：プレイブック、ツール定義、システムプロンプト。`cache_read_input_tokens`で確認し、モデルごとの最小サイズに注意し、変動するバイトをプレフィックスに入れないこと。
4. **出力に規律を**：成果物の仕様としての構造化出力、適正な`max_tokens`、タスクに合わせた`effort`（Haikuには使わない）。
5. **往復を減らし、残りはストリーミング**：直列の呼び出しはすべて、レイテンシと再課金されるコンテキストになります。TTFTはUXの数字です。
6. **並列化は慎重に**：キャッシュをウォームしてから、レート制限の範囲内で展開する。
7. **2速アーキテクチャ**：人間にはインタラクティブレーン、裏のバックログにはBatch API（50%オフ）。プロジェクト単位ではなく、ワークロード単位で決めます。
8. **すべてを精度でゲートする**：最適化の祝勝会の*前に*、品質評価を走らせる。（昨日の評価のセッションが、このラボのもう半分です。）

**ブックマークしておくドキュメント：** platform.claude.com/docs → Prompt caching · Batch processing · Structured outputs · Effort · Streaming · Pricing。